In [9]:
import os
import csv
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader


CSV_PATH = "dataset_split.csv"  
SPLIT_TRAIN = "train"
SPLIT_VAL   = "val"
SPLIT_TEST  = "test"

ORIGINAL_SHAPE = (256, 256)    
DOWNSAMPLE_TO  = (64, 64)        # Downsample frames to this size (width, height)

FRAMES_PER_VIDEO = 120           

# Model / Training settings
INPUT_DIM   = DOWNSAMPLE_TO[0] * DOWNSAMPLE_TO[1] * 3   # e.g., 64*64*3 = 12288
HIDDEN_DIM  = 256
NUM_EPOCHS  = 10
BATCH_SIZE  = 4
LEARNING_RATE = 5e-4


In [10]:

class RawVideoDataset(Dataset):

    def __init__(self, csv_file, split="train", resize_shape=(64, 64)):
        """
        Args:
            csv_file (str): Path to the CSV that has columns: 'filepath', 'label', 'split'.
            split (str): Which split to load ('train', 'val', or 'test').
            resize_shape (tuple): (width, height) to resize each frame.
        """
        self.samples = []
        self.resize_shape = resize_shape
        
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["split"] == split:
                    self.samples.append((row["filepath"], row["label"]))
        
        unique_labels = sorted(list(set([s[1] for s in self.samples])))
        self.label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        

        self.samples = [(fp, self.label_to_idx[lab]) for (fp, lab) in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        """
        Returns:
            frames_tensor: shape (120, input_dim), where input_dim = width*height*3
            label_idx: int, the label of this sequence
        """
        npy_path, label_idx = self.samples[idx]
        
        frames = np.load(npy_path) 
        
        # Downsample + flatten each frame
        processed_frames = []
        for frame in frames:
            resized_frame = cv2.resize(frame, self.resize_shape)  # shape: (64, 64, 3)
            
            
            # Flatten to 1D
            flat_frame = resized_frame.reshape(-1)  # shape: (64*64*3,)
            processed_frames.append(flat_frame)
        
        # Stack into shape (120, input_dim)
        processed_frames = np.array(processed_frames, dtype=np.float32)
        
        # Convert to torch tensor
        frames_tensor = torch.from_numpy(processed_frames)  # shape = (120, input_dim)
        
        return frames_tensor, label_idx



In [14]:


class VideoLSTM(nn.Module):
    """
    A simple LSTM model that takes (batch, seq_len=120, input_dim)
    and outputs a classification over the gesture label.
    """
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(VideoLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=2,  # Stacked LSTMs
                            dropout=0.3,  # Dropout for regularization
                            bidirectional=True,  # Use bidirectional LSTM
                            batch_first=True)
        self.fc   = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        """
        x.shape = (batch, 120, input_dim)
        """
        out, (h_n, c_n) = self.lstm(x)  
        
        last_out = out[:, -1, :]      
        
        logits = self.fc(last_out)     
        return logits


In [ ]:
def train_direct_lstm():
    train_dataset = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_TRAIN, resize_shape=DOWNSAMPLE_TO)
    val_dataset   = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_VAL,   resize_shape=DOWNSAMPLE_TO)
    test_dataset  = RawVideoDataset(csv_file=CSV_PATH, split=SPLIT_TEST,  resize_shape=DOWNSAMPLE_TO)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
    
    num_classes = len(train_dataset.label_to_idx)
    
    model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(NUM_EPOCHS):

        model.train()
        total_loss = 0.0
        for frames_batch, labels_batch in train_loader:

            frames_batch = frames_batch.to(device)
            labels_batch = labels_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(frames_batch) 
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_loader)
        

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for frames_batch, labels_batch in val_loader:
                frames_batch = frames_batch.to(device)
                labels_batch = labels_batch.to(device)
                
                outputs = model(frames_batch)
                loss = criterion(outputs, labels_batch)
                val_loss += loss.item()
                
                _, preds = torch.max(outputs, dim=1)
                correct += (preds == labels_batch).sum().item()
                total += labels_batch.size(0)
        
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_accuracy = (correct / total) if total > 0 else 0
        
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_accuracy*100:.2f}%")
    
    model.eval()
    test_loss, test_correct, test_total = 0.0, 0, 0
    with torch.no_grad():
        for frames_batch, labels_batch in test_loader:
            frames_batch = frames_batch.to(device)
            labels_batch = labels_batch.to(device)
            outputs = model(frames_batch)
            loss = criterion(outputs, labels_batch)
            test_loss += loss.item()
            
            _, preds = torch.max(outputs, dim=1)
            test_correct += (preds == labels_batch).sum().item()
            test_total += labels_batch.size(0)
    avg_test_loss = test_loss / len(test_loader) if len(test_loader) > 0 else 0
    test_accuracy = (test_correct / test_total) if test_total > 0 else 0
    print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy*100:.2f}%")



if __name__ == "__main__":
    train_direct_lstm()


Epoch [1/10] | Train Loss: 1.1036 | Val Loss: 1.0839 | Val Acc: 33.33%
Epoch [2/10] | Train Loss: 1.0214 | Val Loss: 1.2446 | Val Acc: 44.44%
Epoch [3/10] | Train Loss: 0.9519 | Val Loss: 1.1946 | Val Acc: 55.56%
Epoch [4/10] | Train Loss: 0.9544 | Val Loss: 1.2375 | Val Acc: 55.56%
Epoch [5/10] | Train Loss: 0.9364 | Val Loss: 1.1617 | Val Acc: 55.56%
Epoch [6/10] | Train Loss: 0.9017 | Val Loss: 1.2131 | Val Acc: 55.56%
Epoch [7/10] | Train Loss: 0.8520 | Val Loss: 1.3265 | Val Acc: 55.56%
Epoch [8/10] | Train Loss: 0.8528 | Val Loss: 1.2766 | Val Acc: 66.67%
Epoch [9/10] | Train Loss: 0.8131 | Val Loss: 1.3327 | Val Acc: 55.56%
Epoch [10/10] | Train Loss: 0.8092 | Val Loss: 1.3636 | Val Acc: 55.56%
Test Loss: 1.3916, Test Accuracy: 55.56%
